# 02 - Execute DQ Checks (Deterministic Engine)

Runs rules from YAML against CSV datasets and produces results + bad record samples.

In [24]:
from datetime import timezone as dt

print(dt.utc)

UTC


In [64]:
import json
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE = Path(r"../")
DATA_DIR = BASE / "data" / "raw"
RULESET_DIR = BASE / "dq_registry" / "rulesets"
RESULTS_DIR = BASE / "dq_results"
SAMPLES_DIR = RESULTS_DIR / "bad_samples"
RULE_RESULTS = RESULTS_DIR / "rule_results"
RUN_SUMMARIES = RESULTS_DIR / "run_summaries"
RESULTS_DIR.mkdir(exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "materials_master": pd.read_csv(DATA_DIR / "materials_master.csv"),
    "goods_movements": pd.read_csv(DATA_DIR / "goods_movements.csv"),
}


def load_ruleset(name):
    p = RULESET_DIR / f"{name}.yaml"
    return yaml.safe_load(p.read_text(encoding="utf-8"))


mat_ruleset = load_ruleset("materials_master")
gm_ruleset = load_ruleset("goods_movements")

run_id = f"local_run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
print("run_id:", run_id)

run_id: local_run_20260226_203138


## Rule implementations

In [66]:
def is_blank(s):
    if s is None:
        return True
    if isinstance(s, float) and np.isnan(s):
        return True
    return str(s).strip() == ""


def check_schema(df, exp):
    required = exp.get("required_columns", [])
    missing = [c for c in required if c not in df.columns]
    status = "pass" if not missing else "fail"
    observed = {"missing_columns": missing, "required_columns": required}
    return status, observed, None, None


def check_completeness(df, exp):
    col = exp["column"]
    max_null_pct = float(exp.get("max_null_percent", 0))
    total = len(df)
    nulls = int(df[col].apply(is_blank).sum())
    null_pct = (nulls / total * 100) if total else 0.0
    status = "pass" if null_pct <= max_null_pct else "fail"
    observed = {"null_count": nulls, "total": total, "null_pct": null_pct}
    threshold = {"max_null_percent": max_null_pct}
    bad_idx = df.index[df[col].apply(is_blank)]
    return status, observed, threshold, bad_idx


def check_uniqueness(df, exp):
    col = exp["column"]
    max_dup = int(exp.get("max_duplicates_allowed", 0))
    total = len(df)
    dup_counts = df[col].astype(str).value_counts()
    dup_keys = dup_counts[dup_counts > 1].index.tolist()
    duplicates = int(sum(dup_counts.loc[dup_keys] - 1)) if dup_keys else 0
    status = "pass" if duplicates <= max_dup else "fail"
    observed = {"duplicate_keys": dup_keys[:20], "duplicates": duplicates, "total": total}
    threshold = {"max_duplicates_allowed": max_dup}
    bad_idx = df.index[df[col].astype(str).isin(dup_keys)]
    return status, observed, threshold, bad_idx


def check_range(df, exp):
    col = exp["column"]
    minv = exp.get("min", None)
    maxv = exp.get("max", None)
    s = pd.to_numeric(df[col], errors="coerce")
    bad = pd.Series(False, index=df.index)
    if minv is not None:
        bad |= s < float(minv)
    if maxv is not None:
        bad |= s > float(maxv)
    bad |= s.isna()
    failed = int(bad.sum())
    total = len(df)
    status = "pass" if failed == 0 else "fail"
    observed = {"failed": failed, "total": total}
    threshold = {"min": minv, "max": maxv}
    bad_idx = df.index[bad]
    return status, observed, threshold, bad_idx


def check_domain(df, exp):
    col = exp["column"]
    allowed = set(exp.get("allowed_values", []))
    s = df[col].astype(str)
    bad = ~s.isin(list(allowed)) & ~df[col].apply(is_blank)
    failed = int(bad.sum())
    total = len(df)
    status = "pass" if failed == 0 else "fail"
    top_invalid = s[bad].value_counts().head(10).to_dict()
    observed = {"failed": failed, "total": total, "top_invalid_values": top_invalid}
    threshold = {"allowed_values": sorted(list(allowed))}
    bad_idx = df.index[bad]
    return status, observed, threshold, bad_idx


def check_date_not_in_future(df, exp):
    col = exp["column"]
    # parse as date
    parsed = pd.to_datetime(df[col], errors="coerce").dt.date
    today = date.today()
    bad = parsed.isna() | (parsed > today)
    failed = int(bad.sum())
    total = len(df)
    status = "pass" if failed == 0 else "fail"
    observed = {"failed": failed, "total": total, "today": today.isoformat()}
    threshold = {"not_in_future": True}
    bad_idx = df.index[bad]
    return status, observed, threshold, bad_idx


def check_referential_integrity(child_df, exp, datasets):
    child_col = exp["child_column"]
    parent_ds = exp["parent_dataset"]
    parent_col = exp["parent_column"]
    parent_df = datasets[parent_ds]
    parent_keys = set(parent_df[parent_col].astype(str))
    child_keys = child_df[child_col].astype(str)
    bad = (~child_keys.isin(list(parent_keys))) & ~child_df[child_col].apply(is_blank)
    failed = int(bad.sum())
    total = len(child_df)
    status = "pass" if failed == 0 else "fail"
    top_missing = child_keys[bad].value_counts().head(10).to_dict()
    observed = {"failed": failed, "total": total, "top_missing_parent_keys": top_missing}
    threshold = {"parent_dataset": parent_ds, "parent_column": parent_col}
    bad_idx = child_df.index[bad]
    return status, observed, threshold, bad_idx


def check_freshness(df, exp):
    """
    Dataset freshness check:
    PASS if max(ts_column) is within max_age_days from now (UTC).
    """
    ts_col = exp.get("ts_column", "ts_load")
    max_age_days = int(exp.get("max_age_days", -1))

    if ts_col not in df.columns:
        status = "fail"
        observed = {"error": "missing_ts_column", "ts_column": ts_col}
        threshold = {"ts_column": ts_col, "max_age_days": max_age_days}
        bad_idx = None
        return status, observed, threshold, bad_idx

    ts = pd.to_datetime(df[ts_col], errors="coerce", utc=True)

    if ts.notna().sum() == 0:
        # column exists but values are not parseable / all null
        bad = ts.isna()
        status = "fail"
        observed = {
            "error": "ts_column_not_parseable_or_all_null",
            "ts_column": ts_col,
            "not_parseable_or_null_count": int(bad.sum()),
            "total": int(len(df)),
        }
        threshold = {"ts_column": ts_col, "max_age_days": max_age_days}
        bad_idx = df.index[bad]
        return status, observed, threshold, bad_idx

    max_ts = ts.max()
    now_ts = pd.Timestamp.now(tz="UTC")
    age_days = (now_ts - max_ts).total_seconds() / 86400.0

    status = "pass" if age_days <= max_age_days else "fail"
    observed = {
        "max_ts_load_utc": str(max_ts),
        "now_utc": str(now_ts),
        "age_days": round(float(age_days), 4),
    }
    threshold = {"ts_column": ts_col, "max_age_days": max_age_days}

    # Freshness is dataset-level; no row-level bad samples needed
    bad_idx = None
    return status, observed, threshold, bad_idx


CHECKS = {
    "schema": check_schema,
    "completeness": check_completeness,
    "uniqueness": check_uniqueness,
    "range": check_range,
    "domain": check_domain,
    "date_not_in_future": check_date_not_in_future,
    "freshness": check_freshness,
    "referential_integrity": check_referential_integrity,
}

## Execute rulesets and persist results

In [67]:
def save_bad_samples(dataset_id, rule_id, df, bad_idx, max_samples=100):
    if bad_idx is None or len(bad_idx) == 0:
        return None
    sample = df.loc[list(bad_idx)].head(max_samples)
    # Keep small payload (no PII, but still minimize)
    payload_cols = sample.columns[: min(10, len(sample.columns))]
    sample = sample[payload_cols]
    sample_path = SAMPLES_DIR / f"{run_id}__{dataset_id}__{rule_id}.csv"
    sample.to_csv(sample_path, index=False)
    return str(sample_path)


results = []


def run_ruleset(dataset_id, ruleset):
    df = datasets[dataset_id]
    for r in ruleset["rules"]:
        rule_id = r["rule_id"]
        rule_type = r["rule_type"]
        severity = r.get("severity", "medium")
        exp = r.get("expectation", {})
        started = datetime.now(timezone.utc)
        if rule_type in CHECKS:
            if rule_type != "referential_integrity":
                status, observed, threshold, bad_idx = CHECKS[rule_type](df, exp)
            else:
                print(rule_type)
                child_ds = exp["child_dataset"]
                child_df = datasets[child_ds]
                status, observed, threshold, bad_idx = check_referential_integrity(
                    child_df, exp, datasets
                )

        else:
            status, observed, threshold, bad_idx = "fail", {"error": "unknown_rule_type"}, {}, None
        sample_ref = None
        if status == "fail":
            sample_ref = save_bad_samples(dataset_id, rule_id, df, bad_idx, max_samples=100)
        finished = datetime.now(timezone.utc)
        results.append(
            {
                "run_id": run_id,
                "dataset_id": dataset_id,
                "rule_id": rule_id,
                "rule_type": rule_type,
                "severity": severity,
                "status": status,
                "observed_value": json.dumps(observed, ensure_ascii=False),
                "threshold": json.dumps(threshold, ensure_ascii=False),
                "sample_ref": sample_ref,
                "started_at": started.isoformat() + "Z",
                "finished_at": finished.isoformat() + "Z",
                "execution_ms": int((finished - started).total_seconds() * 1000),
            }
        )


run_ruleset("materials_master", mat_ruleset)
run_ruleset("goods_movements", gm_ruleset)

# # Cross-dataset referential integrity
# for r in cross_ruleset["rules"]:
#     exp = r["expectation"]
#     if r["rule_type"] == "referential_integrity":
#         child_ds = exp["child_dataset"]
#         child_df = datasets[child_ds]
#         status, observed, threshold, bad_idx = check_referential_integrity(child_df, exp, datasets)
#         sample_ref = None
#         if status == "fail":
#             sample_ref = save_bad_samples(child_ds, r["rule_id"], child_df, bad_idx, max_samples=100)
#         results.append({
#             "run_id": run_id,
#             "dataset_id": child_ds,
#             "rule_id": r["rule_id"],
#             "rule_type": "referential_integrity",
#             "severity": r.get("severity","critical"),
#             "status": status,
#             "observed_value": json.dumps(observed, ensure_ascii=False),
#             "threshold": json.dumps(threshold, ensure_ascii=False),
#             "sample_ref": sample_ref,
#             "started_at": datetime.now(timezone.utc).isoformat()+"Z",
#             "finished_at": datetime.now(timezone.utc).isoformat()+"Z",
#             "execution_ms": 0,
#         })

res_df = pd.DataFrame(results)
res_path = RULE_RESULTS / f"{run_id}__rule_results.csv"
res_df.to_csv(res_path, index=False)
print("Wrote results:", res_path)
display(res_df.head(50))

referential_integrity
Wrote results: ..\dq_results\rule_results\local_run_20260226_203138__rule_results.csv


,run_id,dataset_id,rule_id,rule_type,severity,status,observed_value,threshold,sample_ref,started_at,finished_at,execution_ms
0,local_run_20260226_203138,materials_master,M001_schema_presence,schema,critical,pass,"{""missing_columns"": [], ""required_columns"": [""...",null,NaN,2026-02-26T20:31:51.994212+00:00Z,2026-02-26T20:31:51.994334+00:00Z,0
1,local_run_20260226_203138,materials_master,M002_pk_unique,uniqueness,critical,fail,"{""duplicate_keys"": [""M00011"", ""M00021""], ""dupl...","{""max_duplicates_allowed"": 0}",..\dq_results\bad_samples\local_run_20260226_2...,2026-02-26T20:31:51.994397+00:00Z,2026-02-26T20:31:52.000734+00:00Z,6
2,local_run_20260226_203138,materials_master,M003_material_id_not_null,completeness,critical,pass,"{""null_count"": 0, ""total"": 202, ""null_pct"": 0.0}","{""max_null_percent"": 0.0}",NaN,2026-02-26T20:31:52.000810+00:00Z,2026-02-26T20:31:52.001723+00:00Z,0
3,local_run_20260226_203138,materials_master,M003_material_name_not_null,completeness,critical,fail,"{""null_count"": 1, ""total"": 202, ""null_pct"": 0....","{""max_null_percent"": 0.0}",..\dq_results\bad_samples\local_run_20260226_2...,2026-02-26T20:31:52.001771+00:00Z,2026-02-26T20:31:52.034980+00:00Z,33
4,local_run_20260226_203138,materials_master,M003_unit_of_measure_not_null,completeness,high,fail,"{""null_count"": 1, ""total"": 202, ""null_pct"": 0....","{""max_null_percent"": 0.0}",..\dq_results\bad_samples\local_run_20260226_2...,2026-02-26T20:31:52.035064+00:00Z,2026-02-26T20:31:52.040201+00:00Z,5
5,local_run_20260226_203138,materials_master,M010_uom_domain,domain,high,pass,"{""failed"": 0, ""total"": 202, ""top_invalid_value...","{""allowed_values"": [""g"", ""kg"", ""l"", ""m"", ""pcs""]}",NaN,2026-02-26T20:31:52.040276+00:00Z,2026-02-26T20:31:52.050157+00:00Z,9
6,local_run_20260226_203138,materials_master,M011_weight_range,range,high,fail,"{""failed"": 1, ""total"": 202}","{""min"": 0, ""max"": 114.24}",..\dq_results\bad_samples\local_run_20260226_2...,2026-02-26T20:31:52.050218+00:00Z,2026-02-26T20:31:52.056422+00:00Z,6
7,local_run_20260226_203138,materials_master,M012_created_at_not_future,date_not_in_future,medium,fail,"{""failed"": 1, ""total"": 202, ""today"": ""2026-02-...","{""not_in_future"": true}",..\dq_results\bad_samples\local_run_20260226_2...,2026-02-26T20:31:52.056495+00:00Z,2026-02-26T20:31:52.067079+00:00Z,10
8,local_run_20260226_203138,materials_master,M003_dataset_is_fresh,freshness,critical,fail,"{""max_ts_load_utc"": ""2026-02-23 00:00:00+00:00...","{""ts_column"": ""ts_load"", ""max_age_days"": 2}",NaN,2026-02-26T20:31:52.067139+00:00Z,2026-02-26T20:31:52.069188+00:00Z,2
9,local_run_20260226_203138,goods_movements,G001_schema_presence,schema,critical,pass,"{""missing_columns"": [], ""required_columns"": [""...",null,NaN,2026-02-26T20:31:52.069412+00:00Z,2026-02-26T20:31:52.069475+00:00Z,0


## Run summary (pass/warn/fail + score)

In [68]:
def summarize(res_df):
    total = len(res_df)
    failed = (res_df["status"] == "fail").sum()
    failed_critical = ((res_df["status"] == "fail") & (res_df["severity"] == "critical")).sum()
    score = max(0.0, 100.0 - 100.0 * (failed / total)) if total else 100.0
    status = "pass"
    if failed_critical > 0:
        status = "fail"
    elif failed > 0:
        status = "warn"
    return {
        "run_id": run_id,
        "status": status,
        "score": round(score, 2),
        "total_rules": total,
        "failed": int(failed),
        "failed_critical": int(failed_critical),
    }


summary = summarize(res_df)
summary_path = RUN_SUMMARIES / f"{run_id}__run_summary.json"
Path(summary_path).write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary, summary_path

({'run_id': 'local_run_20260226_203138',
  'status': 'fail',
  'score': np.float64(52.38),
  'total_rules': 21,
  'failed': 10,
  'failed_critical': 4},
 WindowsPath('../dq_results/run_summaries/local_run_20260226_203138__run_summary.json'))